<a href="https://colab.research.google.com/github/sajina464/C-Programs/blob/main/day_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip -q install -U gradio scikit-learn transformers sentencepiece


In [ ]:

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# Educational sample dataset
data = pd.DataFrame({
    "credit_score": [
        780, 750, 720, 700, 680, 650, 620, 600,
        580, 550, 740, 690, 630, 570, 710, 660,
        790, 610, 670, 540, 730, 640, 590, 560
    ],
    "income": [
        80000, 70000, 60000, 55000, 50000, 45000,
        40000, 35000, 30000, 25000, 65000, 48000,
        42000, 28000, 58000, 44000, 90000, 36000,
        47000, 22000, 62000, 39000, 32000, 26000
    ],
    "debt_ratio": [
        0.15, 0.20, 0.25, 0.30, 0.35, 0.40,
        0.45, 0.50, 0.55, 0.60, 0.22, 0.38,
        0.48, 0.58, 0.28, 0.42, 0.12, 0.52,
        0.36, 0.65, 0.24, 0.46, 0.57, 0.62
    ],
    "late_payments": [
        0, 0, 0, 0, 1, 1, 1, 2,
        2, 3, 0, 1, 2, 3, 0, 1,
        0, 2, 1, 3, 0, 1, 2, 3
    ],
    "risk": [
        0, 0, 0, 0, 0, 1, 1, 1,
        1, 1, 0, 0, 1, 1, 0, 0,
        0, 1, 0, 1, 0, 1, 1, 1
    ]
})

FEATURES = [
    "credit_score", "income",
    "debt_ratio", "late_payments"
]

credit_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

credit_model.fit(data[FEATURES], data["risk"])

print("Credit model trained successfully!")
print("Training records:", len(data))


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

financial_docs = [
    {
        "id": "F1",
        "topic": "Credit Score",
        "source": "Financial Education",
        "text": "Paying bills on time and maintaining a responsible credit history may help improve a credit score."
    },
    {
        "id": "F2",
        "topic": "Credit Utilization",
        "source": "Financial Education",
        "text": "Credit utilization is the revolving credit amount used divided by the total available revolving credit. Lower utilization is generally preferable for credit scoring."
    },
    {
        "id": "F3",
        "topic": "Loan Eligibility",
        "source": "Financial Education",
        "text": "Loan approval depends on lender criteria, credit history, income, existing debt and repayment capacity. A high credit score does not guarantee approval."
    },
    {
        "id": "F4",
        "topic": "Debt-to-Income Ratio",
        "source": "Financial Education",
        "text": "Debt-to-income ratio is monthly debt payments divided by gross monthly income, multiplied by 100."
    },
    {
        "id": "F5",
        "topic": "Budget Planning",
        "source": "Financial Education",
        "text": "A budget tracks income and expenses. Building an emergency fund may help cover unexpected expenses."
    },
    {
        "id": "F6",
        "topic": "Credit Report",
        "source": "Financial Education",
        "text": "Review a credit report regularly to identify inaccurate information. Contact the relevant credit bureau to dispute errors."
    },
    {
        "id": "F7",
        "topic": "Debt Management",
        "source": "Financial Education",
        "text": "Tracking debt payments, avoiding unnecessary borrowing and paying bills on time may help improve financial stability."
    }
]

doc_texts = [doc["text"] for doc in financial_docs]

rag_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

rag_matrix = rag_vectorizer.fit_transform(doc_texts)

def retrieve_documents(question, top_k=3):
    query_vector = rag_vectorizer.transform([question])
    scores = cosine_similarity(query_vector, rag_matrix)[0]
    ranked = scores.argsort()[::-1]

    results = []

    for idx in ranked:
        if scores[idx] <= 0:
            continue

        item = financial_docs[idx].copy()
        item["similarity"] = round(float(scores[idx]), 3)
        results.append(item)

        if len(results) >= top_k:
            break

    return results

print("RAG ready:", len(financial_docs), "documents indexed")


In [ ]:

import torch
from transformers import pipeline

generator = None

try:
    generator = pipeline(
        "text2text-generation",
        model="google/flan-t5-small",
        device=0 if torch.cuda.is_available() else -1
    )
    print("LLM loaded successfully!")
except Exception as e:
    print("LLM loading failed:", str(e))
    print("Fallback evidence-based response will be used.")

print("GPU available:", torch.cuda.is_available())


In [ ]:

def generate_answer(question, credit_info, evidence):
    if not evidence:
        return (
            "Insufficient supporting evidence was found in the "
            "current knowledge base. Please add relevant financial "
            "documents and try again."
        )

    evidence_text = "\n".join(
        f"{doc['id']}: {doc['text']}"
        for doc in evidence
    )

    prompt = f"""
    You are a careful financial education assistant.

    Question: {question}
    Credit model information: {credit_info}
    Supporting evidence:
    {evidence_text}

    Answer in simple English. Use only the supporting evidence
    for financial claims. Do not invent interest rates or guarantee
    loan approval. Explain when evidence is insufficient.
    Mention relevant evidence IDs. This is educational information.

    Answer:
    """

    if generator is None:
        return (
            "Evidence-based explanation:\n\n"
            + "\n\n".join(
                f"{doc['text']} [Evidence: {doc['id']}]"
                for doc in evidence
            )
            + "\n\nThe language model is unavailable, so this is "
              "a retrieved-evidence summary rather than an LLM answer."
        )

    try:
        result = generator(
            prompt,
            max_new_tokens=160,
            do_sample=False,
            truncation=True
        )
        return result[0]["generated_text"]
    except Exception as e:
        return (
            "LLM generation failed. Retrieved evidence is available "
            "below. Error: " + str(e)
        )


def run_financial_assistant(
    question,
    credit_score,
    annual_income,
    debt_ratio_percent,
    late_payments
):
    if not question or not question.strip():
        return "Please enter a question.", "", ""

    try:
        credit_score = float(credit_score)
        annual_income = float(annual_income)
        debt_ratio_percent = float(debt_ratio_percent)
        late_payments = int(late_payments)
    except (ValueError, TypeError):
        return "Please enter valid numeric profile values.", "", ""

    if not 300 <= credit_score <= 850:
        return "Credit score must be between 300 and 850.", "", ""
    if annual_income <= 0:
        return "Annual income must be greater than zero.", "", ""
    if not 0 <= debt_ratio_percent <= 100:
        return "Debt ratio must be between 0 and 100.", "", ""
    if not 0 <= late_payments <= 10:
        return "Late payments must be between 0 and 10.", "", ""

    profile = pd.DataFrame([{
        "credit_score": credit_score,
        "income": annual_income,
        "debt_ratio": debt_ratio_percent / 100,
        "late_payments": late_payments
    }])[FEATURES]

    prediction = int(credit_model.predict(profile)[0])
    risk_probability = float(
        credit_model.predict_proba(profile)[0][1]
    )

    risk_label = (
        "Higher predicted risk"
        if prediction == 1
        else "Lower predicted risk"
    )

    credit_info = (
        f"{risk_label}; model risk score {risk_probability:.1%}. "
        "This is a demonstration model trained on synthetic data."
    )

    evidence = retrieve_documents(question, top_k=3)

    answer = generate_answer(question, credit_info, evidence)

    credit_result = (
        f"### Credit Model Result\n\n"
        f"- **Predicted category:** {risk_label}\n"
        f"- **Model risk score:** {risk_probability:.1%}\n"
        f"- **Credit score entered:** {credit_score:.0f}\n"
        f"- **Annual income entered:** {annual_income:,.0f}\n"
        f"- **Debt ratio entered:** {debt_ratio_percent:.1f}%\n"
        f"- **Late payments entered:** {late_payments}\n\n"
        "This model output is not a validated probability of default "
        "or a loan approval decision."
    )

    if evidence:
        sources = "\n\n".join(
            f"**{doc['id']} — {doc['topic']}**\n"
            f"{doc['text']}\n"
            f"Source label: {doc['source']} | "
            f"Retrieval similarity: {doc['similarity']}"
            for doc in evidence
        )
    else:
        sources = "No relevant evidence found in the knowledge base."

    return answer, credit_result, sources


print("Credit Model + RAG + LLM integration ready!")


In [ ]:

import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 💰 Financial RAG Assistant
        ### Credit Risk Model + RAG + LLM

        Enter a sample credit profile and ask a financial question.
        """
    )

    with gr.Row():
        credit_score = gr.Slider(
            minimum=300,
            maximum=850,
            value=680,
            step=1,
            label="Credit Score"
        )

        annual_income = gr.Number(
            value=50000,
            label="Annual Income (sample currency units)"
        )

    with gr.Row():
        debt_ratio = gr.Slider(
            minimum=0,
            maximum=100,
            value=30,
            step=1,
            label="Debt-to-Income Ratio (%)"
        )

        late_payments = gr.Slider(
            minimum=0,
            maximum=10,
            value=1,
            step=1,
            label="Late Payments"
        )

    question = gr.Textbox(
        label="Ask a Financial Question",
        placeholder="Example: How can I improve my credit score?",
        lines=2
    )

    submit = gr.Button("Analyze", variant="primary")
    clear = gr.ClearButton()

    answer_output = gr.Markdown()
    credit_output = gr.Markdown()
    sources_output = gr.Markdown()

    submit.click(
        fn=run_financial_assistant,
        inputs=[
            question,
            credit_score,
            annual_income,
            debt_ratio,
            late_payments
        ],
        outputs=[
            answer_output,
            credit_output,
            sources_output
        ]
    )

    gr.Examples(
        examples=[
            [
                "How can I improve my credit score?",
                740, 65000, 25, 0
            ],
            [
                "How does debt-to-income ratio work?",
                680, 50000, 40, 1
            ],
            [
                "Does a high credit score guarantee loan approval?",
                790, 80000, 20, 0
            ],
            [
                "How can I manage unexpected expenses?",
                650, 40000, 45, 2
            ]
        ],
        inputs=[
            question,
            credit_score,
            annual_income,
            debt_ratio,
            late_payments
        ]
    )

demo.launch(share=True, debug=True)


/tmp/ipykernel_2568/3741370525.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ab8d4bf1eec1534911.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:

scenarios = pd.DataFrame([
    ["Strong Profile", 790, 90000, 0.15, 0],
    ["Good Profile", 740, 65000, 0.25, 0],
    ["Moderate Profile", 680, 50000, 0.35, 1],
    ["High Debt", 650, 40000, 0.60, 1],
    ["Multiple Late Payments", 600, 35000, 0.50, 3],
    ["Low Credit Score", 550, 28000, 0.65, 3],
    ["High Score High Debt", 780, 70000, 0.70, 2],
    ["Low Income Low Debt", 700, 25000, 0.10, 0]
], columns=[
    "Scenario", "credit_score", "income",
    "debt_ratio", "late_payments"
])

scenario_predictions = credit_model.predict(scenarios[FEATURES])
scenario_probabilities = credit_model.predict_proba(
    scenarios[FEATURES]
)[:, 1]

scenarios["Predicted_Risk"] = [
    "Higher predicted risk" if p == 1
    else "Lower predicted risk"
    for p in scenario_predictions
]

scenarios["Model_Risk_Score_Percent"] = (
    scenario_probabilities * 100
).round(2)

display(scenarios)

scenarios.to_csv(
    "financial_assistant_scenario_report.csv",
    index=False
)

print("Scenario testing completed!")
print("Profiles tested:", len(scenarios))
print("CSV report created successfully!")


In [ ]:

from google.colab import files

files.download("financial_assistant_scenario_report.csv")
